<a href="https://colab.research.google.com/github/syedmahmoodiagents/NLP/blob/main/Simple_RNN_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
corpus = [
    "I love machine learning lot",
    "word2vec is a great algorithm",
    "Implementing word2vec is really fun"
]

In [3]:

sentences = [s.lower().split() for s in corpus]

In [4]:
sentences

[['i', 'love', 'machine', 'learning', 'lot'],
 ['word2vec', 'is', 'a', 'great', 'algorithm'],
 ['implementing', 'word2vec', 'is', 'really', 'fun']]

In [5]:
vocab = sorted(set(word for sent in sentences for word in sent))

In [6]:
vocab

['a',
 'algorithm',
 'fun',
 'great',
 'i',
 'implementing',
 'is',
 'learning',
 'lot',
 'love',
 'machine',
 'really',
 'word2vec']

In [7]:
word2idx = {w:i for i,w in enumerate(vocab)}
# idx2word = {i:w for w,i in word2idx.items()}
idx2word = {i:w for i,w in enumerate(vocab)}

In [8]:
idx2word

{0: 'a',
 1: 'algorithm',
 2: 'fun',
 3: 'great',
 4: 'i',
 5: 'implementing',
 6: 'is',
 7: 'learning',
 8: 'lot',
 9: 'love',
 10: 'machine',
 11: 'really',
 12: 'word2vec'}

In [9]:
vocab_size = len(vocab)

In [10]:
word2idx

{'a': 0,
 'algorithm': 1,
 'fun': 2,
 'great': 3,
 'i': 4,
 'implementing': 5,
 'is': 6,
 'learning': 7,
 'lot': 8,
 'love': 9,
 'machine': 10,
 'really': 11,
 'word2vec': 12}

In [11]:
X = []
Y = []
for sent in sentences:
    for i in range(len(sent)-1):
        X.append(word2idx[sent[i]])
        Y.append(word2idx[sent[i+1]])

In [12]:
X = torch.tensor(X)
Y = torch.tensor(Y)

In [13]:
print(X)
print(Y)

tensor([ 4,  9, 10,  7, 12,  6,  0,  3,  5, 12,  6, 11])
tensor([ 9, 10,  7,  8,  6,  0,  3,  1, 12,  6, 11,  2])


In [14]:
X.shape

torch.Size([12])

In [15]:
XX = X.reshape(3,4)

In [18]:
XX

tensor([[ 4,  9, 10,  7],
        [12,  6,  0,  3],
        [ 5, 12,  6, 11]])

In [16]:
YY = Y.reshape(3,4)

In [17]:
vocab_size

13

In [26]:

class NextWordRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x) # [3, 4, 100]
        out, hx = self.rnn(x) # [3, 4, 16], [1, 3, 16]
        out = out[:, -1, :] # [3, 16]
        out = self.fc(out) # [3, 10]
        return out

In [35]:
model = NextWordRNN(13)

In [32]:
outputs = model(XX)

In [36]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [40]:
for epoch in range(300):
    optimizer.zero_grad()
    outputs = model(XX)
    loss = loss_fn(outputs, YY[:, -1])
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 3.2385356426239014
Epoch: 50 Loss: 0.020768919959664345
Epoch: 100 Loss: 0.009921747259795666
Epoch: 150 Loss: 0.0063309031538665295
Epoch: 200 Loss: 0.0044470313005149364
Epoch: 250 Loss: 0.003325687488541007


In [41]:
def predict_next(word):
    model.eval()
    idx = torch.tensor([[word2idx[word.lower()]]])

    with torch.no_grad():
        out = model(idx)
        pred = torch.argmax(out).item()

    return idx2word[pred]

In [42]:
print("machine=>", predict_next("machine"))
print("is=>", predict_next("is"))
print("word2vec=>", predict_next("word2vec"))

machine=> fun
is=> fun
word2vec=> fun
